# UCI Multiple Features (6 views): lcc vs multiview spectral clustering

- load benchmark dataset
- run lcc discovery algorithm
- run multiview spectral clustering baseline 
- print nmi comparison table

In [1]:
import sys
from pathlib import Path
import numpy as np

def _find_coderepo_root() -> Path:
    here = Path.cwd().resolve()
    candidates = [here] + list(here.parents)
    for p in candidates:
        if (p / 'LCCdiscovery.py').exists():
            return p
        if (p / 'codeRepo' / 'LCCdiscovery.py').exists():
            return p / 'codeRepo'
    raise FileNotFoundError('could not locate codeRepo (LCCdiscovery.py) from current working directory')

code_repo = _find_coderepo_root()
sys.path.insert(0, str(code_repo))

from LCCdiscovery import LCCConfig, discover_lcc

from dataclasses import dataclass
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.cluster import SpectralClustering
from sklearn.neighbors import NearestNeighbors
import pandas as pd

In [2]:
@dataclass
class BenchResult:
    method: str
    n: int
    n_views: int
    n_true_classes: int
    nmi_all: float
    nmi_non_noise: float | None
    n_lccs: int | None = None
    n_noise: int | None = None

def _preprocess_views(Xs: list[np.ndarray], *, pca_dim: int | None = 50, random_state: int = 0) -> list[np.ndarray]:
    Xs_proc: list[np.ndarray] = []
    for X in Xs:
        X2 = StandardScaler(with_mean=True, with_std=True).fit_transform(X)
        if pca_dim is not None and X2.shape[1] > pca_dim:
            X2 = PCA(n_components=pca_dim, random_state=random_state).fit_transform(X2)
        Xs_proc.append(X2.astype(float))
    return Xs_proc

def _labels_from_components(n: int, components: list[list[int]]) -> np.ndarray:
    labels = np.full(n, -1, dtype=int)
    for ci, comp in enumerate(components):
        labels[np.asarray(comp, dtype=int)] = ci
    return labels

def _nmi_lcc(y_true: np.ndarray, components: list[list[int]]) -> tuple[float, float | None]:
    y_pred = _labels_from_components(len(y_true), components)
    nmi_all = float(normalized_mutual_info_score(y_true, y_pred, average_method='arithmetic'))
    mask = y_pred >= 0
    if int(mask.sum()) >= 2 and len(np.unique(y_true[mask])) >= 2:
        nmi_non_noise = float(normalized_mutual_info_score(y_true[mask], y_pred[mask], average_method='arithmetic'))
    else:
        nmi_non_noise = None
    return nmi_all, nmi_non_noise

def _distance_matrices(Xs: list[np.ndarray], *, metric: str = 'euclidean') -> list[np.ndarray]:
    return [pairwise_distances(X, metric=metric).astype(np.float32, copy=False) for X in Xs]

def _uci_multifeature(download_dir: Path) -> tuple[list[np.ndarray], np.ndarray]:
    import urllib.request
    def _download(url: str, dest: Path) -> Path:
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() and dest.stat().st_size > 0:
            return dest
        print('downloading:', url)
        urllib.request.urlretrieve(url, dest)
        return dest

    base = 'https://archive.ics.uci.edu/ml/machine-learning-databases/mfeat/'
    files = ['mfeat-fou', 'mfeat-fac', 'mfeat-kar', 'mfeat-pix', 'mfeat-zer', 'mfeat-mor']
    paths = [_download(base + f, download_dir / f) for f in files]
    Xs = [np.loadtxt(p, dtype=float) for p in paths]
    n = int(Xs[0].shape[0])
    if n != 2000:
        raise ValueError(f'unexpected n={n}; expected 2000 for mfeat')
    y = np.repeat(np.arange(10, dtype=int), 200)
    return Xs, y

def run_lcc(
    *,
    Xs: list[np.ndarray],
    y: np.ndarray,
    alpha_best: float,
    pca_dim: int | None = 50,
    distance_metric: str = 'euclidean',
    method: str = 'normal',
    fdr: str = 'bh',
    min_component_size: int = 3,
    random_state: int = 0,
) -> BenchResult:
    cfg = LCCConfig(alpha=float(alpha_best), method=method, fdr=fdr, min_component_size=int(min_component_size))
    Xs_proc = _preprocess_views(Xs, pca_dim=pca_dim, random_state=random_state)
    dist_mats = _distance_matrices(Xs_proc, metric=distance_metric)
    lcc = discover_lcc(dist_mats, cfg)
    nmi_all, nmi_non_noise = _nmi_lcc(y, lcc.components)
    return BenchResult(
        method=f"lcc (alpha={float(alpha_best):.4f}, metric={distance_metric})",
        n=int(len(y)),
        n_views=int(len(Xs)),
        n_true_classes=int(len(np.unique(y))),
        nmi_all=float(nmi_all),
        nmi_non_noise=nmi_non_noise,
        n_lccs=int(len(lcc.components)),
        n_noise=int(len(lcc.noise)),
    )

def run_multiview_spectral(
    *,
    Xs: list[np.ndarray],
    y: np.ndarray,
    n_clusters: int,
    n_neighbors: int = 20,
    pca_dim: int | None = 50,
    random_state: int = 0,
) -> BenchResult:
    Xs_proc = _preprocess_views(Xs, pca_dim=pca_dim, random_state=random_state)
    n = int(len(y))
    A = np.zeros((n, n), dtype=np.float32)
    for X in Xs_proc:
        nn = NearestNeighbors(n_neighbors=int(n_neighbors), metric='euclidean')
        nn.fit(X)
        idx = nn.kneighbors(return_distance=False)
        Ai = np.zeros((n, n), dtype=np.float32)
        rows = np.repeat(np.arange(n, dtype=int), int(n_neighbors))
        cols = idx.reshape(-1)
        Ai[rows, cols] = 1.0
        Ai = np.maximum(Ai, Ai.T)
        np.fill_diagonal(Ai, 0.0)
        A += Ai
    A /= float(len(Xs_proc))
    A = np.maximum(A, 0.0)
    np.fill_diagonal(A, 0.0)

    sc = SpectralClustering(
        n_clusters=int(n_clusters),
        affinity='precomputed',
        assign_labels='kmeans',
        random_state=int(random_state),
    )
    y_pred = sc.fit_predict(A)
    nmi = float(normalized_mutual_info_score(y, y_pred, average_method='arithmetic'))
    return BenchResult(
        method=f"multiview spectral (euclidean, fused knn={int(n_neighbors)}, k={int(n_clusters)})",
        n=int(len(y)),
        n_views=int(len(Xs)),
        n_true_classes=int(len(np.unique(y))),
        nmi_all=float(nmi),
        nmi_non_noise=None,
    )

In [3]:
#load benchmark
uci_dir = (code_repo / '.cache' / 'uci_mfeat')
Xs_uci, y_uci = _uci_multifeature(uci_dir)
print('views:', len(Xs_uci), 'n:', Xs_uci[0].shape[0], 'classes:', len(np.unique(y_uci)))

alpha_best = 0.01

views: 6 n: 2000 classes: 10


In [4]:
res_lcc = run_lcc(
    Xs=Xs_uci,
    y=y_uci,
    alpha_best=alpha_best,
    pca_dim=50,
    distance_metric='euclidean',
    method='normal',
    fdr='bh',
    min_component_size=3,
    random_state=0,
)
res_lcc

BenchResult(method='lcc (alpha=0.0100, metric=euclidean)', n=2000, n_views=6, n_true_classes=10, nmi_all=0.6056779467727523, nmi_non_noise=0.7681717485697848, n_lccs=6, n_noise=410)

## results

In [5]:
#run multiview spectral clustering baseline
res_spec = run_multiview_spectral(
    Xs=Xs_uci,
    y=y_uci,
    n_clusters=10,
    n_neighbors=20,
    pca_dim=50,
    random_state=0,
)

#comparison table
df = pd.DataFrame([res_lcc.__dict__, res_spec.__dict__])
df

,method,n,n_views,n_true_classes,nmi_all,nmi_non_noise,n_lccs,n_noise
0,"lcc (alpha=0.0100, metric=euclidean)",2000,6,10,0.605678,0.768172,6.0,410.0
1,"multiview spectral (euclidean, fused knn=20, k...",2000,6,10,0.843159,NaN,NaN,NaN
